# Aula 04 - Preparação e Tratamento de Dados - Parte 1 (Limpeza, Nulos e Outliers)

**Disciplina:** Aprendizado de Máquina e Redes Neurais  
**Professor:** Eduardo Lázaro Roesler de Oliveira  
**Instituição:** UNIVEM — Centro Universitário de Marília  

---

> [!NOTE]
> 🔙 **De onde viemos:** Na Aula 03, aprendemos a manipular, filtrar e agrupar tabelas com Pandas. No entanto, nenhum dado da vida real vem perfeito: nos deparamos com buracos, cadastros duplicados e números absurdos que distorcem as médias.
> 🎯 **Objetivo Principal da Aula:** Diagnosticar e tratar falhas de integridade em tabelas com Pandas: identificar e preencher valores ausentes (`NaN` com média/mediana), remover linhas duplicadas e detectar valores aberrantes (*outliers*) usando a regra do Intervalo Interquartil (IQR).
> 🚀 **Para onde vamos:** Na próxima aula ('Preparação e Tratamento de Dados - Parte 2'), resolveremos o próximo obstáculo: a nossa tabela está limpa, mas ainda contém colunas de texto ('Masculino', 'São Paulo', 'Ouro') que os modelos matemáticos não conseguem calcular, além de colunas com escalas de tamanho totalmente desproporcionais.

---

## Organização Tática da Aula

| Módulo | Atividade | Foco Pedagógico |
| :--- | :--- | :--- |
| **Módulo 1** | **Fundamentação Teórica & Diagnóstico de Dados** | O impacto de dados ruidosos em ML, tipos de dados nulos e a Regra do IQR para outliers. |
| **Módulo 2** | **Estratégias de Imputação vs. Remoção** | Diagnosticar antes de alterar: duplicatas, nulos e escolhas entre `.dropna()` e `.fillna()`. |
| **Módulo 3** | **Prática Guiada no Google Colab** | 8 blocos de código em Python minuciosamente comentados linha por linha com caixas de dúvidas. |
| **Módulo 4** | **Prática Orientada & Experimentação Fácil** | Higienização de dados e detecção de "Cheaters" (outliers) em games online. |
| **Módulo 5** | **Checklist de Autonomia & Bibliografia** | Autoavaliação do estudante e referências oficiais. |

---

## Módulo 1: Fundamentação Teórica & Diagnóstico de Dados

### 1.1 O Princípio "Garbage In, Garbage Out" (GIGO)
Em Machine Learning, os algoritmos são essencialmente otimizadores matemáticos. Se fornecermos a eles um conjunto de dados contaminado por valores ausentes (`NaN`), duplicatas inconsistentes ou *outliers* aberrantes, o modelo aprenderá padrões distorcidos.

Estima-se que Cientistas de Dados dediquem de **60% a 80% do tempo de um projeto** exclusivamente às etapas de coleta, limpeza e preparação dos dados!

```mermaid
graph LR
    A["🗑️ DADOS RUÍDOSOS<br/>(Nulos / Outliers)"] --> B["⚙️ LIMPEZA E ETL<br/>(Dropna / Fillna / IQR)"]
    B --> C["✨ DADOS HIGIENIZADOS<br/>(Limpos e Confiáveis)"]
    C --> D["🧠 MODELO DE IA<br/>(Previsões de Alta Acurácia)"]
```

> [!TIP]
> 👾 **Analogia Geek — Detectando Cheaters / Hackers em Jogos Online:**
> Imagine que em um jogo de tiro online (como Counter-Strike ou Valorant) a média de abates por partida de um jogador normal seja entre 10 e 30. De repente, surge uma conta nova com **3.500 abates por partida**! Esse valor discrepante é um **Outlier** (e provavelmente um Cheater/Hacker). Se colocarmos esse dado no nosso modelo sem limpar, ele vai achar que 3.500 abates é algo normal e arruinar a pontuação de todos!

> [!NOTE]
> 💡 **Curiosidade da Aula — O Bug do Mariner 1 (Garbage In, Garbage Out):**
> Em **1962, a NASA lançou a sonda Mariner 1** em direção a Vênus. Poucos minutos após a decolagem, o foguete se desviou do curso e teve que ser destruído no ar. A causa? Um **único caractere traço (`-`) ausente** na especificação dos dados de entrada do software! Esse foi um dos erros de dados mais caros da história da ciência (custou mais de 80 milhões de dólares na época) e demonstrou a importância vital de validar e limpar os dados de entrada!  
> 🔗 **Documentação Pandas:** [pandas.DataFrame.fillna](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html)

> [!IMPORTANT]
> 💡 **Em 1 Frase:** O princípio GIGO estabelece que se inserirmos dados corrompidos ou ruidosos na entrada, o modelo de IA produzirá previsões erradas e não confiáveis na saída.

---

## Módulo 2: O Mecanismo por Dentro & Regras Práticas

### 2.1 Principais Métodos de Limpeza do Pandas

| Comando Pandas | O que faz no dataset | Quando utilizar |
| :--- | :--- | :--- |
| `df.isnull().sum()` | Conta a quantidade exata de `NaN` por coluna. | Diagnóstico inicial do dataset. |
| `df.dropna(subset=['col'])` | Deleta as linhas onde a coluna possui `NaN`. | Quando os nulos representam < 5% da base. |
| `df['col'].fillna(mediana)` | Preenche os `NaN` de uma coluna com a mediana. | Para preservar o tamanho do dataset sem distorcer. |
| `df.duplicated().sum()` | Identifica a presença de linhas exatamente idênticas. | Antes de qualquer treinamento para evitar viés. |
| `df.drop_duplicates()` | Remove as linhas repetidas mantendo a primeira ocorrência. | Limpeza obrigatória em pipelines de ETL. |

> [!IMPORTANT]
> 💡 **Em 1 Frase:** Preencher nulos com a Mediana preserva a quantidade total de linhas da tabela sem inflacionar as estatísticas por causa de valores extremos.

---

## Módulo 3: Prática Guiada no Google Colab

Abra o [Google Colab](https://colab.research.google.com), crie um novo Notebook e acompanhe os blocos de código abaixo.

> [!NOTE]
> **Roteiro da limpeza:** primeiro olhamos a tabela, depois contamos os problemas, removemos duplicatas, decidimos como tratar nulos e só então investigamos valores muito fora do padrão. Execute um bloco por vez; cada resultado será usado no próximo.

---

### Bloco 3.1 — Gerando um Dataset Ruidoso Simulado de Jogadores

> [!IMPORTANT]
> 🤔 **Dúvidas Comuns de Iniciantes:**  
> **O que é o valor `np.nan`?**  
> Significa *Not a Number* (Não é um Número). É a forma como o Python representa um dado que está faltando ou em branco em uma tabela.

> [!TIP]
> Antes de executar, procure `np.nan` no dicionário. Em qual coluna há um dado ausente? A resposta aparecerá novamente no diagnóstico do próximo bloco.

In [1]:
# 1. Importamos as bibliotecas Pandas e NumPy
import pandas as pd
import numpy as np

# 2. Criamos um dicionário contendo falhas propositais: duplicatas, nulos e um outlier
dicionario_jogadores_ruidoso = {
    'player_id': [101, 102, 103, 104, 105, 102, 107], # ID 102 é DUPLICADO
    'nickname': ['Gamer1', 'Shadow', 'Dragon', 'Shadow', 'Valkyrie', 'Shadow', np.nan],
    'horas_jogadas': [45, 120, np.nan, 120, 80, 120, 30], # Um valor NaN em horas
    'pontos_partida': [1500, 3200, 2100, 3200, 2800, 3200, 999999] # 999999 é OUTLIER (Cheater)!
}

# 3. Convertemos o dicionário em um DataFrame Pandas
tabela_jogadores = pd.DataFrame(dicionario_jogadores_ruidoso)

# 4. Exibimos a tabela com ruídos no console
print("--- DATASET ORIGINAL COM NULOS, DUPLICATAS E CHEATER ---")
print(f"📐 Dimensão Inicial: {tabela_jogadores.shape[0]} linhas x {tabela_jogadores.shape[1]} colunas")
print("\nTabela com ruidos:")
print(tabela_jogadores)

--- DATASET ORIGINAL COM NULOS, DUPLICATAS E CHEATER ---
📐 Dimensão Inicial: 7 linhas x 4 colunas

Tabela com ruidos:
   player_id  nickname  horas_jogadas  pontos_partida
0        101    Gamer1           45.0            1500
1        102    Shadow          120.0            3200
2        103    Dragon            NaN            2100
3        104    Shadow          120.0            3200
4        105  Valkyrie           80.0            2800
5        102    Shadow          120.0            3200
6        107       NaN           30.0          999999


---

### Bloco 3.2 — Diagnóstico Inicial de Valores Nulos (`isnull().sum()`)

> [!IMPORTANT]
> 🤔 **Dúvidas Comuns de Iniciantes:**  
> **O que faz a combinação `.isnull().sum()`?**  
> O `.isnull()` cria uma tabela de `True` (onde é nulo) e `False` (onde tem dado). O `.sum()` soma todos os `True` de cada coluna, revelando exatamente quantos nulos existem.

In [2]:
# 1. Contamos a quantidade exata de valores nulos (NaN) em cada coluna da tabela
contagem_nulos_por_coluna = tabela_jogadores.isnull().sum()

# 2. Exibimos o relatório de nulos por coluna
print("--- DIAGNÓSTICO DE VALORES AUSENTES (NaN) ---")
print("Valores Ausentes por Coluna:\n", contagem_nulos_por_coluna)

--- DIAGNÓSTICO DE VALORES AUSENTES (NaN) ---
Valores Ausentes por Coluna:
 player_id         0
nickname          1
horas_jogadas     1
pontos_partida    0
dtype: int64


---

### Bloco 3.3 — Detecção e Remoção de Duplicatas (`duplicated`)

> [!IMPORTANT]
> 🤔 **Dúvidas Comuns de Iniciantes:**  
> **O que faz o parâmetro `keep='first'`?**  
> Ele instrui o Pandas a manter a primeira vez que a linha apareceu na tabela e deletar apenas as repetições que vierem depois dela.

In [3]:
# 1. Identificamos e contamos quantas linhas são exatamente idênticas a outras
quantidade_duplicatas = tabela_jogadores.duplicated().sum()

# 2. Removemos as linhas duplicadas mantendo a primeira ocorrência
tabela_sem_duplicatas = tabela_jogadores.drop_duplicates(keep='first').copy()

# 3. Exibimos o resultado no console
print("--- REMOÇÃO DE REGISTROS DUPLICADOS ---")
print(f"⚠️ Total de linhas duplicadas encontradas: {quantidade_duplicatas}")
print(f"✅ Duplicatas removidas! Linhas restantes na tabela: {len(tabela_sem_duplicatas)}")

--- REMOÇÃO DE REGISTROS DUPLICADOS ---
⚠️ Total de linhas duplicadas encontradas: 1
✅ Duplicatas removidas! Linhas restantes na tabela: 6


---

### Bloco 3.4 — Remoção Direta de Nulos com `.dropna()`

> [!IMPORTANT]
> 🤔 **Dúvidas Comuns de Iniciantes:**  
> **Por que NUNCA devemos sair usando `.dropna()` sem pensar?**  
> Porque se a sua base for pequena ou tiver nulos espalhados em várias colunas, o `.dropna()` pode apagar metade da sua tabela e destruir informações valiosas!

In [4]:
# 1. Removemos qualquer linha que contenha pelo menos um valor NaN em qualquer coluna
tabela_sem_nulos_direto = tabela_sem_duplicatas.dropna()

# 2. Exibimos quantas linhas restaram após a exclusão brutal
print("--- DEMONSTRAÇÃO DA REMOÇÃO DIRETA COM DROPNA ---")
print(f"Linhas restantes após dropna(): {len(tabela_sem_nulos_direto)}")
print(tabela_sem_nulos_direto)

--- DEMONSTRAÇÃO DA REMOÇÃO DIRETA COM DROPNA ---
Linhas restantes após dropna(): 4
   player_id  nickname  horas_jogadas  pontos_partida
0        101    Gamer1           45.0            1500
1        102    Shadow          120.0            3200
3        104    Shadow          120.0            3200
4        105  Valkyrie           80.0            2800


---

### Bloco 3.5 — Imputação Estatística de Nulos (`fillna` com Mediana)

> [!IMPORTANT]
> 🤔 **Dúvidas Comuns de Iniciantes:**  
> **Por que usamos a Mediana em vez da Média para preencher números nulos?**  
> Porque a Média é facilmente "puxada" para cima por um valor gigante (outlier). A **Mediana** pega exatamente o valor central da lista ordenada e não se deixa enganar por números malucos!

In [5]:
# 1. Calculamos a mediana da coluna 'horas_jogadas' (ignora automaticamente os nulos)
valor_mediana_horas = tabela_sem_duplicatas['horas_jogadas'].median()

# 2. Preenchemos os nulos da coluna com a mediana calculada
tabela_sem_duplicatas['horas_jogadas'] = tabela_sem_duplicatas['horas_jogadas'].fillna(valor_mediana_horas)

# 3. Exibimos a tabela com o valor preenchido
print("--- IMPUTAÇÃO DE VALORES NULOS COM MEDIANA ---")
print(f"📊 Mediana calculada para preenchimento: {valor_mediana_horas} horas")
print("\nTabela após imputação da Mediana:")
print(tabela_sem_duplicatas[['player_id', 'nickname', 'horas_jogadas']])

--- IMPUTAÇÃO DE VALORES NULOS COM MEDIANA ---
📊 Mediana calculada para preenchimento: 80.0 horas

Tabela após imputação da Mediana:
   player_id  nickname  horas_jogadas
0        101    Gamer1           45.0
1        102    Shadow          120.0
2        103    Dragon           80.0
3        104    Shadow          120.0
4        105  Valkyrie           80.0
6        107       NaN           30.0


---

### Bloco 3.6 — Imputação Categórica com a Moda (`fillna` com Moda)

> [!IMPORTANT]
> 🤔 **Dúvidas Comuns de Iniciantes:**  
> **O que é a Moda (`.mode()`)?**  
> É o valor que **mais se repete** em uma coluna. Como não existe "média" para nomes ou palavras, usamos o apelido mais comum (Moda) para preencher textos vazios.

In [6]:
# 1. Calculamos o apelido/texto mais frequente (Moda) da coluna 'nickname'
valor_moda_nickname = tabela_sem_duplicatas['nickname'].mode()[0]

# 2. Preenchemos o apelido ausente com a Moda calculada
tabela_sem_duplicatas['nickname'] = tabela_sem_duplicatas['nickname'].fillna(valor_moda_nickname)

# 3. Exibimos o resultado
print("--- IMPUTAÇÃO CATEGÓRICA COM MODA ---")
print(f"🗣️ Apelido mais comum (Moda): {valor_moda_nickname}")

--- IMPUTAÇÃO CATEGÓRICA COM MODA ---
🗣️ Apelido mais comum (Moda): Shadow


---

### Bloco 3.7 — Cálculo dos Quartis (Q1, Q3) e do IQR

> [!IMPORTANT]
> 🤔 **Dúvidas Comuns de Iniciantes:**  
> **O que é o IQR (Interquartile Range)?**  
> É a distância entre os 25% mais baixos ($Q_1$) e os 75% mais altos ($Q_3$) dos seus dados. O IQR mede onde está concentrado o "miolo" normal das informações.

> [!NOTE]
> Este é um primeiro contato com IQR. O objetivo não é memorizar a fórmula, mas acompanhar o processo: encontrar o intervalo usual dos pontos e usar um limite para investigar valores muito distantes.

In [7]:
# 1. Calculamos o primeiro quartil (25%) e o terceiro quartil (75%) da pontuação
quartil_1 = tabela_sem_duplicatas['pontos_partida'].quantile(0.25)
quartil_3 = tabela_sem_duplicatas['pontos_partida'].quantile(0.75)

# 2. Calculamos a Amplitude Interquartil (IQR = Q3 - Q1)
amplitude_iqr = quartil_3 - quartil_1

# 3. Exibimos os valores estatísticos calculados
print("--- CÁLCULO ESTATÍSTICO DO IQR PARA OUTLIERS ---")
print(f"Q1 (25% dos dados): {quartil_1}")
print(f"Q3 (75% dos dados): {quartil_3}")
print(f"Amplitude IQR:      {amplitude_iqr}")

--- CÁLCULO ESTATÍSTICO DO IQR PARA OUTLIERS ---
Q1 (25% dos dados): 2275.0
Q3 (75% dos dados): 3200.0
Amplitude IQR:      925.0


---

### Bloco 3.8 — Remoção de Outliers via Limite Superior do IQR

> [!IMPORTANT]
> 🤔 **Dúvidas Comuns de Iniciantes:**  
> **Por que a fórmula do limite usa `1.5 * IQR`?**  
> É uma regra estatística clássica proposta por John Tukey para sinalizar valores que merecem investigação. Um valor acima de $Q_3 + 1.5 \times IQR$ é um candidato a outlier; antes de removê-lo, devemos avaliar se é erro, fraude ou um caso real importante.

In [8]:
# 1. Definimos o limite superior máximo aceitável para pontuações normais
limite_superior_aceitavel = quartil_3 + 1.5 * amplitude_iqr

# 2. Filtramos a tabela mantendo apenas pontuações menores ou iguais ao limite aceitável
tabela_limpa_final = tabela_sem_duplicatas[tabela_sem_duplicatas['pontos_partida'] <= limite_superior_aceitavel].copy()

# 3. Exibimos a tabela higienizada sem o jogador hacker
print("--- FILTRAGEM DO JOGADOR HACKER (OUTLIER) ---")
print(f"🚨 Limite Superior Aceitável de Pontos: {limite_superior_aceitavel}")
print("\n🎉 TABELA FINAL COMPLETAMENTE HIGIENIZADA E SEM OUTLIERS:")
print(tabela_limpa_final)

--- FILTRAGEM DO JOGADOR HACKER (OUTLIER) ---
🚨 Limite Superior Aceitável de Pontos: 4587.5

🎉 TABELA FINAL COMPLETAMENTE HIGIENIZADA E SEM OUTLIERS:
   player_id  nickname  horas_jogadas  pontos_partida
0        101    Gamer1           45.0            1500
1        102    Shadow          120.0            3200
2        103    Dragon           80.0            2100
3        104    Shadow          120.0            3200
4        105  Valkyrie           80.0            2800


---

## Módulo 4: Prática Orientada & Experimentação Fácil

### Roteiro de Personalização para o Estudante:
Altere o código de remoção de outliers do placar do jogo abaixo para personalizar o filtro:

1. **Altere o fator do multiplicador do IQR:** O padrão é `1.5`. Na linha `fator_multiplicador_iqr = 1.5`, mude para `1.0` ou `2.0`.
2. **Re-execute e observe:** Veja se o seu filtro elimina o jogador hacker sem apagar os jogadores de pontuação normal!

In [9]:
# =============================================================================
# CÓDIGO BASE PRONTO PARA SUA EXPERIMENTAÇÃO
# =============================================================================
import pandas as pd

# 1. Tabela sintética com placar de jogos
tabela_placar_games = pd.DataFrame({
    'jogador': ['ProPlayer', 'Noob123', 'Hacker99', 'PlayerX', 'GamerBR'],
    'pontos': [4500, 1200, 950000, 3800, 2900] # 950.000 é um outlier evidente!
})

# -----------------------------------------------------------------------------
# STEP 1: PASSO DE EXPERIMENTAÇÃO DO ALUNO — ALTERE O FATOR MULTIPLICADOR DO IQR:
# -----------------------------------------------------------------------------
fator_multiplicador_iqr = 1.5 # Tente mudar para 1.0 ou 2.0 e veja o limite!

# 2. Calculamos os quartis e o limite superior
quartil_1 = tabela_placar_games['pontos'].quantile(0.25)
quartil_3 = tabela_placar_games['pontos'].quantile(0.75)
amplitude_iqr = quartil_3 - quartil_1

limite_superior_calculado = quartil_3 + fator_multiplicador_iqr * amplitude_iqr

# 3. Filtramos mantendo apenas pontuações normais
tabela_sem_hacker = tabela_placar_games[tabela_placar_games['pontos'] <= limite_superior_calculado]

# 4. Exibimos o placar higienizado
print(f"--- PLACAR HIGIENIZADO SEM HACKERS (Fator IQR = {fator_multiplicador_iqr}) ---")
print(tabela_sem_hacker)

--- PLACAR HIGIENIZADO SEM HACKERS (Fator IQR = 1.5) ---
     jogador  pontos
0  ProPlayer    4500
1    Noob123    1200
3    PlayerX    3800
4    GamerBR    2900


---

### 🔮 Aquecimento & Spoiler da Próxima Aula (Para Ir Além)

> [!TIP]
> 🧠 **Conceito-Semente — Como somar 'São Paulo' com R$ 10.000?**  
> Os algoritmos de Machine Learning são equações matemáticas. Eles não sabem o que é a palavra 'Feminino' ou a categoria 'Premium'. Na **Aula 05**, aprenderemos **Encoding** (como transformar palavras em códigos binários $0$ e $1$) e **Scaling** (como colocar uma coluna de Salário de R$ 50.000 e uma coluna de Idade de 20 anos na mesma régua para que o salário não engula a idade).
> 
> 🚀 **Desafio Proativo de Autoestudo (Opcional):**  
> Se você tem as categorias `Bronze`, `Prata` e `Ouro`, faz sentido transformá-las em `1, 2, 3`? E se as categorias fossem `Carro`, `Moto`, `Caminhão`, transformar em `1, 2, 3` seria justo ou criaria uma ordem inventada? Reflita sobre isso!

---

## Módulo 5: Checklist de Autonomia do Estudante

- [ ] Compreendi a importância de tratar dados ruidosos antes do treinamento de ML.
- [ ] Sei diagnosticar nulos com `df.isnull().sum()`.
- [ ] Sei quando aplicar a remoção (`dropna`) ou imputação com Mediana/Moda (`fillna`).
- [ ] Dominei o uso do `drop_duplicates()`.
- [ ] Consigo alterar o fator do IQR e observar como o limite de investigação muda.

---

## Referências Bibliográficas & Documentações Oficiais

- 📖 **Livro Texto:** Géron, A. — *Mãos à Obra: Aprendizado de Máquina com Scikit-Learn, Keras & TensorFlow* (Capítulo 2: Limpeza de Dados).
- 🔗 **Documentação Pandas:** [pandas Working with missing data](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html)

---

# 🧪 Extensão Prática Avançada: Laboratório Técnico com Dados Reais (`pratica_real.md`)

> 🔬 **Sobre esta Extensão:**  
> Esta seção adicional consolida o aprendizado com uma abordagem **mais avançada, direta e profissional**, sem dados sintéticos e utilizando datasets reais consagrados da Ciência de Dados.


# Laboratório Prático — Aula 04: Higienização de Dados e Tratamento de Outliers

**Disciplina:** Aprendizado de Máquina e Redes Neurais  
**Professor:** Eduardo Lázaro Roesler de Oliveira  
**Instituição:** UNIVEM — Centro Universitário de Marília  
**Dataset Real:** *Titanic Dataset* (891 passageiros com falhas reais de cadastro, valores nulos e tarifas discrepantes)

---

## 🎯 Objetivo do Laboratório
Executar o diagnóstico e a limpeza de uma base de dados real: mapear percentuais de nulos por coluna, aplicar imputação condicional de valores ausentes pela mediana, remover colunas com comprometimento severo e identificar/filtrar outliers de tarifas usando a Regra do Intervalo Interquartil (IQR).

---

### Passo 1 — Diagnóstico Percentual de Dados Faltantes (`NaN`)

In [10]:
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Carregamos o dataset real do Titanic
df_titanic = sns.load_dataset('titanic')

# 2. Calculamos a quantidade e o percentual exato de nulos por coluna
relatorio_nulos = pd.DataFrame({
    'Total_Nulos': df_titanic.isnull().sum(),
    'Percentual_%': (df_titanic.isnull().sum() / len(df_titanic) * 100).round(2)
})

print("--- RELATÓRIO DE DADOS AUSENTES ---")
print(relatorio_nulos[relatorio_nulos['Total_Nulos'] > 0])

--- RELATÓRIO DE DADOS AUSENTES ---
             Total_Nulos  Percentual_%
age                  177         19.87
embarked               2          0.22
deck                 688         77.22
embark_town            2          0.22


---

### Passo 2 — Tomada de Decisão: Descarte vs Imputação

In [11]:
# 1. Coluna 'deck' tem mais de 77% de nulos -> Descartamos a coluna inteira
df_limpo = df_titanic.drop(columns=['deck']).copy()

# 2. Coluna 'embarked' tem apenas 2 nulos -> Preenchemos com a Moda (porto mais frequente)
porto_mais_comum = df_limpo['embarked'].mode()[0]
df_limpo['embarked'] = df_limpo['embarked'].fillna(porto_mais_comum)

# 3. Coluna 'age' tem 19.8% de nulos -> Imputamos pela Mediana de idade por Classe
mediana_idade_por_classe = df_limpo.groupby('pclass')['age'].transform('median')
df_limpo['age'] = df_limpo['age'].fillna(mediana_idade_por_classe)

print(f"Nulos restantes em 'age': {df_limpo['age'].isnull().sum()}")
print(f"Nulos restantes em 'embarked': {df_limpo['embarked'].isnull().sum()}")

Nulos restantes em 'age': 0
Nulos restantes em 'embarked': 0


---

### Passo 3 — Detecção de Outliers na Tarifa (`fare`) via Regra do IQR

In [12]:
# 1. Calculamos o Primeiro Quartil (Q1) e Terceiro Quartil (Q3)
Q1 = df_limpo['fare'].quantile(0.25)
Q3 = df_limpo['fare'].quantile(0.75)
IQR = Q3 - Q1

# 2. Definimos os limites inferior e superior para outliers
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1 (25%): {Q1:.2f} | Q3 (75%): {Q3:.2f} | IQR: {IQR:.2f}")
print(f"Limite de Corte Superior para Outliers: R$ {limite_superior:.2f}")

# 3. Contamos quantos passageiros pagaram tarifas consideradas aberrantes
outliers_tarifa = df_limpo[df_limpo['fare'] > limite_superior]
print(f"Total de registros identificados como Outliers de Tarifa: {len(outliers_tarifa)} ({len(outliers_tarifa)/len(df_limpo)*100:.2f}%)")

Q1 (25%): 7.91 | Q3 (75%): 31.00 | IQR: 23.09
Limite de Corte Superior para Outliers: R$ 65.63
Total de registros identificados como Outliers de Tarifa: 116 (13.02%)


---

### Passo 4 — Filtragem e Comparação Estatística

In [13]:
# 1. Geramos o dataset final sem os outliers de tarifa extrema
df_sem_outliers = df_limpo[df_limpo['fare'] <= limite_superior].copy()

print("--- COMPARAÇÃO DA MÉDIA E DESVIO PADRÃO ---")
print(f"Média original da Tarifa:     R$ {df_limpo['fare'].mean():.2f} (Desvio: {df_limpo['fare'].std():.2f})")
print(f"Média sem Outliers da Tarifa: R$ {df_sem_outliers['fare'].mean():.2f} (Desvio: {df_sem_outliers['fare'].std():.2f})")

--- COMPARAÇÃO DA MÉDIA E DESVIO PADRÃO ---
Média original da Tarifa:     R$ 32.20 (Desvio: 49.69)
Média sem Outliers da Tarifa: R$ 17.82 (Desvio: 13.58)


---

## 🏆 Desafio Técnico de Validação
Verifique se a variável `age` (idade) contém outliers após a imputação aplicando a mesma fórmula do IQR (`Q3 + 1.5 * IQR`).

In [14]:
# =============================================================================
# ESPAÇO PARA RESOLUÇÃO DO DESAFIO TÉCNICO
# Implemente seu código abaixo para validar o desafio proposto:
# =============================================================================
